# COMP53415 – Assembly of 2D Puzzles
## Learning Example

**File:** `username-assembly-train.ipynb`

⚠️ **Very Important** ⚠️ Only code inside the Jupyter Notebooks will be evaluated. Only put code in .py files that is not important to your method. The code provides a dataloader in a Python file. If you significantly change the dataloader logic, then move the code into the Jupyter Notebook.

This notebook is intended for **training or fitting any learning-based components** of your puzzle-solving system (e.g., small heads on top of frozen pretrained backbones).

If your method is purely classical (no learning), you may **leave this notebook mostly empty** or use it only for small experiments.

> Replace `username` in the file name with your CIS username before submission.



## 0. Time Script (Do not remove)
This cell records the start time of the entire script.

We use this timestamp to measure how long the notebook takes to run from start to finish. This is useful for: 
 - Comparing training speed across different model sizes
 - Understanding the computational cost of different design choices
 - Debugging performance issues

The value stored in script_start is used later to compute the total runtime.

⚠️ Important rules:
 - Do not remove this cell.
 - Do not change the variable name script_start.
 - Do not move this cell lower in the notebook.

If you modify or rerun this cell after training has started, the reported runtime will be incorrect.

The code below captures a high-resolution timestamp using time.perf_counter().

In [ ]:
import time
from typing import Final
script_start: Final = time.perf_counter() # Do not remove or change this value

## 1. Setup

Edit this cell to configure your environment (paths, seeds, device selection, etc.).


In [3]:
# Standard library utilities for filesystem access, archive handling (ZIP/TAR),
# text parsing, JSON metadata, and basic timing/math utilities.
# These are typically used when loading datasets packaged as archives
# and normalising them into a common on-disk structure.
import os, io, re, zipfile, tarfile, json, math, time
from pathlib import Path
from collections import defaultdict

# Core numerical and image-processing libraries.
# NumPy underpins almost all geometric and numerical operations,
# while PIL is used for loading, saving, and manipulating fragment images.
import numpy as np
from PIL import Image

# PyTorch core library and neural network components.
# These imports support defining models that predict fragment pose
# (e.g. x, y position and rotation) and training them end-to-end.
import torch
import torch.nn as nn
import torch.nn.functional as F

# Dataset and DataLoader abstractions.
# These are used to stream large fragment collections efficiently,
# either as standard indexed datasets or iterable datasets for
# very large or on-the-fly generated puzzles.
from torch.utils.data import IterableDataset, DataLoader
from torch.utils.data import Dataset, DataLoader

# Lightweight data containers and typing utilities.
# Dataclasses are often used to store structured metadata
# (e.g. fragment attributes, puzzle configuration),
# while typing improves code clarity in complex pipelines.
from dataclasses import dataclass
from typing import Any, Dict, List, Optional, Tuple, Union

# Image preprocessing and augmentation utilities.
# These are typically applied to fragment images before passing
# them through a neural network (normalisation, resizing, cropping).
from torchvision import transforms

# Python-side randomness utilities.
# Used for lightweight sampling or shuffling when NumPy or PyTorch
# RNGs are not required.
import random

# Progress bar utility.
# Commonly wrapped around training loops or dataset preprocessing
# to give real-time feedback on long-running operations.
from tqdm import tqdm

# OpenCV computer vision library.
# Frequently used for contour extraction, edge detection,
# morphological operations, and geometric reasoning on fragments.
import cv2

# Utility for visualising batches of images as a single grid.
# Useful for quickly inspecting fragment sets or model inputs/outputs.
from torchvision.utils import make_grid

# Duplicate imports are intentionally retained so that this cell
# can be copy-pasted or partially reused without dependency issues.
import numpy as np
from PIL import Image

# Visualisation library for qualitative inspection and debugging.
# Common uses include plotting fragment crops, contours,
# intermediate feature maps, and reconstructed layouts.
import matplotlib.pyplot as plt

In [4]:
# Paths and dataset locations
DATA_ROOT = Path("../dataset")          # Root folder containing all datasets
COCOTILES_ZIP = DATA_ROOT / "CocoTiles.zip"
DAFNE_ZIP     = DATA_ROOT / "Dafne.zip"
OUTPUT_MODEL = "example_username-model.pth"

# Reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

# Dataset and augmentation settings
AUGMENTATION = "None"                 # Options: "None", "Simple", "Moderate", "Hard"
FRAGMENT_FIXED_SIZE = (24, 24)        # (width, height) in pixels
NORMALIZE_RGB = True

# Training and model configuration
EPOCHS = 10
LEARNING_RATE=3e-4
NUMBER_WORKERS=0
BATCH_SIZE=4
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


DATASET_ZIP=COCOTILES_ZIP
DATASET_MODE= 'None' # Note can be simple ...

# Configuration summary (sanity check)
print("DATA_ROOT:      ", DATA_ROOT)
print("COCOTILES_ZIP:  ", COCOTILES_ZIP)
print("DAFNE_ZIP:      ", DAFNE_ZIP)
print("AUGMENTATION:   ", AUGMENTATION)
print("EPOCHS:         ", EPOCHS)
print("FRAGMENT_SIZE:  ", FRAGMENT_FIXED_SIZE)
print("NORMALIZE_RGB:  ", NORMALIZE_RGB)
print("DEVICE:         ", DEVICE)
print("================")
print("RUNNING ON      ",DATASET_ZIP)


DATA_ROOT:       ..\dataset
COCOTILES_ZIP:   ..\dataset\CocoTiles.zip
DAFNE_ZIP:       ..\dataset\Dafne.zip
AUGMENTATION:    None
EPOCHS:          10
FRAGMENT_SIZE:   (24, 24)
NORMALIZE_RGB:   True
DEVICE:          cuda
RUNNING ON       ..\dataset\CocoTiles.zip


## 2. Data Loading (Optional)

If you are training any models, you can add your **data loading utilities** here.

You may choose to:
- Load MS-COCO tile patches as training samples.
- Load DAFNE fragment crops or descriptors.
- Build small datasets for training lightweight heads on top of **frozen** pretrained backbones.

Remember:
- You must **not** use any external labelled data (COCO labels, segmentation maps, etc.).
- Only frozen model-zoo backbones may be used as feature extractors.


In [5]:
from UnifiedPuzzleSetZipDataset import make_unified_puzzle_dataloader_zip, batch_fragments_from_collate

## 3. Model Definition (Optional)

Define any **trainable components** of your system here (e.g., small MLPs, attention layers, or other heads on top of frozen pretrained features).

All **trainable parameters at inference** must sum to **≤ 15M**.

In [6]:
import torch
import torch.nn as nn
import math

# ----------------------------
# Piece encoder: CNN -> vector
# ----------------------------
class SmallCNNEncoder(nn.Module):
    """
    Encodes each fragment (RGBA) into a feature vector.
    Input : [B*N, 4, H, W]
    Output: [B*N, d_model]
    """
    def __init__(self, in_ch: int = 4, d_model: int = 128, dropout: float = 0.0):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, 32, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Dropout2d(p=dropout),
        )
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.proj = nn.Linear(128, d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h = self.conv(x)
        h = self.pool(h).flatten(1)   # [B*N, 128]
        return self.proj(h)           # [B*N, d_model]


# ----------------------------
# Global pooling helpers (mask-aware)
# ----------------------------
def masked_mean(x: torch.Tensor, valid_mask: torch.Tensor, eps: float = 1e-6) -> torch.Tensor:
    """
    x: [B, N, D]
    valid_mask: [B, N] bool, True where VALID (not pad)
    returns: [B, D]
    """
    m = valid_mask.unsqueeze(-1).type_as(x)      # [B,N,1]
    s = (x * m).sum(dim=1)                       # [B,D]
    denom = m.sum(dim=1).clamp_min(eps)          # [B,1]
    return s / denom

def masked_max(x: torch.Tensor, valid_mask: torch.Tensor) -> torch.Tensor:
    """
    x: [B, N, D]
    valid_mask: [B, N] bool, True where VALID (not pad)
    returns: [B, D]
    """
    # set invalid positions to -inf so they never win the max
    neg_inf = torch.finfo(x.dtype).min
    x2 = x.masked_fill(~valid_mask.unsqueeze(-1), neg_inf)
    return x2.max(dim=1).values


# ----------------------------
# Model: CNN -> GlobalPool -> Per-piece MLP
# ----------------------------
class PuzzlePoseModel(nn.Module):
    """
    PointNet-style:
      per-piece features f_i
      global feature g = pool_i(f_i)
      per-piece prediction from [f_i || g]

    Predicts: [x, y, rot_deg] per fragment.
    """
    def __init__(
        self,
        d_model: int = 128,
        dropout: float = 0.1,
        pool: str = "max",   # "max", "mean", or "maxmean"
    ):
        super().__init__()
        assert pool in {"max", "mean", "maxmean"}
        self.pool = pool

        self.cnn = SmallCNNEncoder(in_ch=4, d_model=d_model, dropout=dropout)

        g_dim = d_model if pool in {"max", "mean"} else 2 * d_model
        self.head = nn.Sequential(
            nn.LayerNorm(d_model + g_dim),
            nn.Linear(d_model + g_dim, d_model),
            nn.ReLU(inplace=True),
            nn.Dropout(p=dropout),
            nn.Linear(d_model, 3),
        )

    def forward(self, imgs: torch.Tensor, pad_mask: torch.Tensor) -> torch.Tensor:
        """
        imgs: [B, N, 4, H, W]
        pad_mask: [B, N] True where PAD
        returns: [B, N, 3]
        """
        B, N, C, H, W = imgs.shape
        x = imgs.view(B * N, C, H, W)
        f = self.cnn(x).view(B, N, -1)           # [B,N,D]

        valid = ~pad_mask                         # [B,N] True where VALID

        if self.pool == "max":
            g = masked_max(f, valid)              # [B,D]
        elif self.pool == "mean":
            g = masked_mean(f, valid)             # [B,D]
        else:  # "maxmean"
            g = torch.cat([masked_max(f, valid), masked_mean(f, valid)], dim=-1)  # [B,2D]

        g_rep = g.unsqueeze(1).expand(B, N, g.shape[-1])    # [B,N,g_dim]
        h = torch.cat([f, g_rep], dim=-1)                   # [B,N,D+g_dim]
        pred = self.head(h)                                  # [B,N,3]
        return pred


## 4. Training Loop

Implement your training loop here if you are using learning-based components.

You should:
- Log losses and relevant metrics.
- Respect the **parameter** and **runtime** constraints.
- Save any trained weights you intend to load in your test notebook as `username-assembly-model.pth`.


In [7]:
# ----------------------------
# Loss + metrics
# ----------------------------
@dataclass
class LossWeights:
    xy: float = 1.0
    rot: float = 0.2

def masked_smooth_l1(pred: torch.Tensor, target: torch.Tensor, mask_valid: torch.Tensor) -> torch.Tensor:
    """
    pred/target: [B,N,...]
    mask_valid: [B,N] True for valid positions
    """
    # Broadcast mask to pred shape
    while mask_valid.ndim < pred.ndim:
        mask_valid = mask_valid.unsqueeze(-1)
    diff = F.smooth_l1_loss(pred, target, reduction="none")
    diff = diff * mask_valid.to(diff.dtype)
    denom = mask_valid.to(diff.dtype).sum().clamp_min(1.0)
    return diff.sum() / denom

@torch.no_grad()
def compute_metrics(pred_xy: torch.Tensor, pred_rot: torch.Tensor,
                    gt_xy: torch.Tensor, gt_rot: torch.Tensor,
                    valid_mask: torch.Tensor) -> Dict[str, float]:
    """
    valid_mask: [B,N] True for valid.
    """
    # XY L2
    xy_err = (pred_xy - gt_xy).pow(2).sum(dim=-1).sqrt()  # [B,N]
    xy_err = xy_err[valid_mask]
    # Rotation absolute error in degrees (no wrap handling here; add wrap if needed)
    rot_err = (pred_rot - gt_rot).abs()
    rot_err = rot_err[valid_mask]

    out = {}
    out["xy_rmse"] = float(torch.sqrt((xy_err.pow(2).mean() if xy_err.numel() else torch.tensor(0.0)).cpu()))
    out["xy_mae"]  = float((xy_err.mean() if xy_err.numel() else torch.tensor(0.0)).cpu())
    out["rot_mae_deg"] = float((rot_err.mean() if rot_err.numel() else torch.tensor(0.0)).cpu())
    return out

In [7]:
def run_one_epoch(
    model: nn.Module,
    loader,
    optimizer: Optional[torch.optim.Optimizer],
    device: DEVICE,
    loss_w: LossWeights,
    grad_clip: float = 1.0,
    epoch: int = 0,
    epochs: int = 0,
    log_every: int = 25,          # print every N steps
    log_prefix: str = "",         # "train" or "val"
) -> Dict[str, float]:
    is_train = optimizer is not None
    model.train(is_train)

    total_loss = 0.0
    total_xy_loss = 0.0
    total_rot_loss = 0.0
    n_steps = 0

    # metric accumulators
    m_xy_rmse = 0.0
    m_xy_mae = 0.0
    m_rot_mae = 0.0

    # For speed/ETA style logging
    start_t = time.time()
    last_log_t = start_t

    # Try to get total steps (works for standard DataLoaders)
    try:
        total_steps = len(loader)
    except TypeError:
        total_steps = None

    for step_idx, batch in enumerate(loader, start=1):
        pack = batch_fragments_from_collate(batch, device=device)
        imgs = pack["imgs"]
        gt_xy = pack["xy"]
        gt_rot = pack["rot"]
        pad_mask = pack["pad_mask"]
        valid_mask = ~pad_mask  # True where real fragments exist

        if imgs.numel() == 0:
            continue

        if is_train:
            optimizer.zero_grad(set_to_none=True)

        pred = model(imgs, pad_mask=pad_mask)  # [B,N,3]
        pred_xy = pred[..., :2]
        pred_rot = pred[..., 2]

        xy_loss = masked_smooth_l1(pred_xy, gt_xy, valid_mask)
        rot_loss = masked_smooth_l1(pred_rot, gt_rot, valid_mask)
        loss = loss_w.xy * xy_loss + loss_w.rot * rot_loss

        if is_train:
            loss.backward()
            if grad_clip is not None and grad_clip > 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=grad_clip)
            optimizer.step()

        # metrics
        metrics = compute_metrics(pred_xy, pred_rot, gt_xy, gt_rot, valid_mask)

        total_loss += float(loss.detach().cpu())
        total_xy_loss += float(xy_loss.detach().cpu())
        total_rot_loss += float(rot_loss.detach().cpu())
        m_xy_rmse += metrics["xy_rmse"]
        m_xy_mae += metrics["xy_mae"]
        m_rot_mae += metrics["rot_mae_deg"]
        n_steps += 1

        # ----------------------------
        # Iteration progress printing
        # ----------------------------
        if log_every and (step_idx % log_every == 0):
            now = time.time()
            dt = now - last_log_t
            elapsed = now - start_t
            steps_per_s = (log_every / dt) if dt > 0 else float("inf")

            # Running averages
            avg_loss = total_loss / n_steps
            avg_xy = total_xy_loss / n_steps
            avg_rot = total_rot_loss / n_steps
            avg_xy_rmse = m_xy_rmse / n_steps
            avg_rot_mae = m_rot_mae / n_steps

            # LR (for AdamW there is usually 1 param group)
            lr_val = None
            if optimizer is not None and optimizer.param_groups:
                lr_val = optimizer.param_groups[0].get("lr", None)

            if total_steps is not None:
                prog = f"{step_idx:04d}/{total_steps:04d}"
            else:
                prog = f"{step_idx:04d}/????"

            e_str = f"{epoch:03d}/{epochs:03d}" if (epoch and epochs) else ""

            lr_str = f" | lr {lr_val:.2e}" if lr_val is not None else ""
            prefix = f"{log_prefix} " if log_prefix else ""
            epoch_str = f"Epoch {e_str} | " if e_str else ""

            print(
                f"{prefix}{epoch_str}iter {prog}"
                f"{lr_str} | "
                f"loss {avg_loss:.4f} (xy {avg_xy:.4f}, rot {avg_rot:.4f}) | "
                f"xy_rmse {avg_xy_rmse:.3f}px | rot_mae {avg_rot_mae:.3f}deg | "
                f"{steps_per_s:.2f} it/s | {elapsed:.1f}s elapsed"
            )

            last_log_t = now

    if n_steps == 0:
        return {
            "loss": 0.0, "xy_loss": 0.0, "rot_loss": 0.0,
            "xy_rmse": 0.0, "xy_mae": 0.0, "rot_mae_deg": 0.0
        }

    return {
        "loss": total_loss / n_steps,
        "xy_loss": total_xy_loss / n_steps,
        "rot_loss": total_rot_loss / n_steps,
        "xy_rmse": m_xy_rmse / n_steps,
        "xy_mae": m_xy_mae / n_steps,
        "rot_mae_deg": m_rot_mae / n_steps,
    }
    
def fit(
    zip_path: str,
    fixed_size: Tuple[int, int] = (128, 128),
    normalize_rgb: bool = True,
    batch_size: int = 2,
    num_workers: int = 4,
    epochs: int = 20,
    lr: float = 3e-4,
    weight_decay: float = 1e-4,
    device: Optional[str] = None,
    d_model: int = 256,
    nhead: int = 8,
    num_layers: int = 4,
    augment_mode='None',
    loss_w: LossWeights = LossWeights(xy=1.0, rot=0.2),
):
    device = torch.device(device or ("cuda" if torch.cuda.is_available() else "cpu"))

    # Your provided maker
    train_loader = make_unified_puzzle_dataloader_zip(
        zip_path=zip_path,
        split="train",
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        fixed_size=fixed_size,
        normalize_rgb=normalize_rgb,
        return_optional_images=False,
        augment_mode=augment_mode
    )
    
    val_loader = make_unified_puzzle_dataloader_zip(
        zip_path=zip_path,
        split="val",
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        fixed_size=fixed_size,
        normalize_rgb=normalize_rgb,
        return_optional_images=False,
    )

    model = PuzzlePoseModel(
        d_model=d_model
    ).to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    # A reasonable default scheduler: cosine with warmup-ish via OneCycle also works, but keep simple:
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(1, epochs))

    best_val = float("inf")
    best_state = None

    for epoch in range(1, epochs + 1):
        train_stats = run_one_epoch(
            model, train_loader, optimizer, device,
            loss_w=loss_w, grad_clip=1.0,
            epoch=epoch, epochs=epochs,
            log_every=25, log_prefix="train"
        )
    
        model.eval()
        with torch.no_grad():
            val_stats = run_one_epoch(
                model, val_loader, optimizer=None, device=device,
                loss_w=loss_w,
                epoch=epoch, epochs=epochs,
                log_every=50, log_prefix="val"
            )
    
        scheduler.step()
    
        print(
            f"Epoch {epoch:03d}/{epochs} | "
            f"train loss {train_stats['loss']:.4f} (xy {train_stats['xy_loss']:.4f}, rot {train_stats['rot_loss']:.4f}) | "
            f"val loss {val_stats['loss']:.4f} (xy {val_stats['xy_loss']:.4f}, rot {val_stats['rot_loss']:.4f}) | "
            f"val xy_rmse {val_stats['xy_rmse']:.3f} px | val rot_mae {val_stats['rot_mae_deg']:.3f} deg"
        )

    if best_state is not None:
        model.load_state_dict(best_state)
        print(f"Loaded best model (val loss {best_val:.4f}).")

    return model, optimizer,scheduler,best_val

# ----------------------------
# Example call (edit zip_path)
# ----------------------------
model,optimizer,scheduler,best_val = fit(
    zip_path=DATASET_ZIP,
    fixed_size=FRAGMENT_FIXED_SIZE,
    normalize_rgb=NORMALIZE_RGB,
    batch_size=BATCH_SIZE,
    num_workers=NUMBER_WORKERS,
    epochs=EPOCHS,
    lr=LEARNING_RATE,
    augment_mode=DATASET_MODE
)


train Epoch 001/010 | iter 0025/1250 | lr 3.00e-04 | loss 0.1238 (xy 0.1222, rot 0.0078) | xy_rmse 0.479px | rot_mae 0.100deg | 5.08 it/s | 4.9s elapsed
train Epoch 001/010 | iter 0050/1250 | lr 3.00e-04 | loss 0.1047 (xy 0.1037, rot 0.0050) | xy_rmse 0.446px | rot_mae 0.076deg | 14.65 it/s | 6.6s elapsed
train Epoch 001/010 | iter 0075/1250 | lr 3.00e-04 | loss 0.0977 (xy 0.0969, rot 0.0037) | xy_rmse 0.433px | rot_mae 0.063deg | 15.83 it/s | 8.2s elapsed
train Epoch 001/010 | iter 0100/1250 | lr 3.00e-04 | loss 0.0940 (xy 0.0934, rot 0.0030) | xy_rmse 0.427px | rot_mae 0.057deg | 14.97 it/s | 9.9s elapsed
train Epoch 001/010 | iter 0125/1250 | lr 3.00e-04 | loss 0.0918 (xy 0.0913, rot 0.0026) | xy_rmse 0.423px | rot_mae 0.052deg | 14.99 it/s | 11.5s elapsed
train Epoch 001/010 | iter 0150/1250 | lr 3.00e-04 | loss 0.0903 (xy 0.0899, rot 0.0023) | xy_rmse 0.420px | rot_mae 0.048deg | 16.44 it/s | 13.1s elapsed
train Epoch 001/010 | iter 0175/1250 | lr 3.00e-04 | loss 0.0892 (xy 0.0888

## 5. Save Model (best)

In [8]:
def save_checkpoint(
    path: str,
    model: torch.nn.Module,
    optimizer: Optional[torch.optim.Optimizer] = None,
    scheduler: Optional[torch.optim.lr_scheduler._LRScheduler] = None,
    epoch: Optional[int] = None,
    best_val: Optional[float] = None,
    extra: Optional[Dict[str, Any]] = None,
):
    """
    Save a training checkpoint.

    Args:
        path: output .pt or .pth file
        model: model to save
        optimizer: optimizer (optional)
        scheduler: LR scheduler (optional)
        epoch: current epoch (optional)
        best_val: best validation loss so far (optional)
        extra: any additional metadata to store
    """

    ckpt = {
        "model_state": model.state_dict(),
    }

    if optimizer is not None:
        ckpt["optimizer_state"] = optimizer.state_dict()
    if scheduler is not None:
        ckpt["scheduler_state"] = scheduler.state_dict()
    if epoch is not None:
        ckpt["epoch"] = epoch
    if best_val is not None:
        ckpt["best_val"] = best_val
    if extra is not None:
        ckpt["extra"] = extra

    torch.save(ckpt, path)
    print(f"Checkpoint saved to: {path}")


save_checkpoint(
    path=OUTPUT_MODEL,
    model=model,
    optimizer=optimizer,
    scheduler=scheduler,
    epoch=EPOCHS,
    best_val=best_val,
)

Checkpoint saved to: example_username-model.pth


## Qualitative Visualisation

In [8]:
import numpy as np
from PIL import Image

def assemble_canvas(
    imgs_norm,
    xy,
    rot_deg,
    pad_mask,
    img_sizes,
    canvas_size=(240, 240),
    background=(255, 255, 255, 255),
    assume_xy_is_center=True,
):
    """
    Assemble fragments into a canvas in the same coordinate system as xy.

    imgs_norm: [N,C,H,W] or [B,N,C,H,W] in [0,1] (after denormalize)
    xy:        [N,2] or [B,N,2] in reference pixels
    img_sizes  [N,2] original fragment sizes
    rot_deg:   [N] or [B,N]
    pad_mask:  [N] or [B,N]
    pad_mask:  [N,2] or [B,N,2]
    """

    # ---- tensors -> numpy ----
    imgs = imgs_norm.detach().cpu().numpy()
    xy = xy.detach().cpu().numpy()
    rot_deg = rot_deg.detach().cpu().numpy()
    pad_mask = pad_mask.detach().cpu().numpy()
    img_sizes = img_sizes.detach().cpu().numpy()

    # ---- unwrap batch if present ----
    if imgs.ndim == 5: imgs = imgs[0]
    if xy.ndim == 3: xy = xy[0]
    if rot_deg.ndim == 2: rot_deg = rot_deg[0]
    if pad_mask.ndim == 2: pad_mask = pad_mask[0]

    N = imgs.shape[0]

    # ----------------------------
    # Convert images to PIL RGBA (and upscale to target_tile_size)
    # ----------------------------
    pil_imgs = []
    for i in range(N):
        img = imgs[i]  # [C,H,W] likely

        # CHW -> HWC
        if img.ndim == 3 and img.shape[0] in (1, 3, 4):
            img = np.transpose(img, (1, 2, 0))

        # Ensure [0,1]
        if img.min() < 0:
            img = (img + 1) / 2

        img = np.clip(img * 255, 0, 255).astype(np.uint8)

        # Ensure RGBA
        if img.shape[-1] == 3:
            alpha = np.full((*img.shape[:2], 1), 255, dtype=np.uint8)
            img = np.concatenate([img, alpha], axis=-1)
        elif img.shape[-1] == 4:
            pass
        else:
            raise ValueError(f"Expected 3 or 4 channels after conversion, got {img.shape}")

        pil = Image.fromarray(img, mode="RGBA")

        img_size = img_sizes[i]

        # Upscale to original size
        if img_size is not None:
            pil = pil.resize((img_size[0], img_size[1]), resample=Image.NEAREST)

        pil_imgs.append(pil)

    # ----------------------------
    # Create canvas
    # ----------------------------
    W, H = canvas_size
    canvas = Image.new("RGBA", (W, H), background)

    # ----------------------------
    # Composite fragments
    # ----------------------------
    for i in range(N):
        if pad_mask[i]:
            continue

        img = pil_imgs[i]
        x, y = float(xy[i, 0]) * W, float(xy[i, 1]) * H
        theta = float(rot_deg[i])

        w, h = img.size
        cx, cy = w / 2, h / 2

        # Rotate around centre
        rot_img = img.rotate(-theta, resample=Image.BICUBIC, expand=True)
        rw, rh = rot_img.size
        dx = rw / 2 - cx
        dy = rh / 2 - cy

        # If xy is centre coords, convert to top-left for pasting
        if assume_xy_is_center:
            paste_x = int(round(x - cx - dx)) 
            paste_y = int(round(y - cy - dy)) 
        else:
            paste_x = int(round(x - dx)) 
            paste_y = int(round(y - dy)) 

        canvas.alpha_composite(rot_img, (paste_x, paste_y))

    return canvas

# ----------------------------
# Reload training dataloader too
# ----------------------------
train_loader = make_unified_puzzle_dataloader_zip(
    zip_path=DATASET_ZIP,
    split="train",
    batch_size=1,
    shuffle=False,
    num_workers=0,
    fixed_size=FRAGMENT_FIXED_SIZE,
    normalize_rgb=NORMALIZE_RGB,
    return_optional_images=False,
    augment_mode="None"
)
val_loader = make_unified_puzzle_dataloader_zip(
    zip_path=DATASET_ZIP,
    split="val",
    batch_size=1,
    shuffle=False,
    num_workers=0,
    fixed_size=FRAGMENT_FIXED_SIZE,
    normalize_rgb=NORMALIZE_RGB,
    return_optional_images=False,
    augment_mode="None"
)

NUM_TRAIN_EXAMPLES = 1
NUM_VAL_EXAMPLES = 1

# --------------------------------
# RGB de-normalisation (ImageNet-style)
# --------------------------------
RGB_MEAN = torch.tensor([0.485, 0.456, 0.406])
RGB_STD  = torch.tensor([0.229, 0.224, 0.225])

def denormalize_imgs(imgs: torch.Tensor) -> torch.Tensor:
    """
    imgs: [B,N,C,H,W] where C is 3 (RGB) or 4 (RGBA)
          If C==4, assumes channel 3 is alpha and leaves it unchanged.
    returns: same shape, RGB in [0,1] (alpha preserved if present)
    """
    assert imgs.ndim == 5, f"Expected [B,N,C,H,W], got {tuple(imgs.shape)}"
    B, N, C, H, W = imgs.shape
    if C not in (3, 4):
        raise ValueError(f"Expected C=3 or C=4, got C={C}")

    mean = RGB_MEAN.view(1, 1, 3, 1, 1).to(device=imgs.device, dtype=imgs.dtype)
    std  = RGB_STD.view(1, 1, 3, 1, 1).to(device=imgs.device, dtype=imgs.dtype)

    out = imgs.clone()
    out[..., :3, :, :] = (out[..., :3, :, :] * std + mean).clamp(0.0, 1.0)

    # If alpha exists, you typically want it in [0,1] for display as well
    if C == 4:
        out[..., 3:4, :, :] = out[..., 3:4, :, :].clamp(0.0, 1.0)

    return out

def render_examples(loader, split_name: str, num_examples: int):
    shown = 0
    with torch.no_grad():
        for batch in loader:
            pack = batch_fragments_from_collate(batch, device=DEVICE)
            

            imgs = pack["imgs"]          # [1,N,C,H,W]
            gt_xy = pack["xy"]           # [1,N,2]
            gt_rot = pack["rot"]         # [1,N]
            pad_mask = pack["pad_mask"]  # [1,N]
            img_sizes = pack["img_sizes"]  # [1,N,2]

            pred = model(imgs, pad_mask=pad_mask)
            pred_xy = pred[..., :2]
            pred_rot = pred[..., 2]


            canvas_width = int(pack["meta"][0]["xy_canvas_W"])
            canvas_height = int(pack["meta"][0]["xy_canvas_H"])

            imgs_norm = denormalize_imgs(imgs) # Remove the normalisation that occurs in dataloader

            print("imgs_norm:", imgs_norm.shape)        # [B,N,C,H,W]
            print("gt_xy:", gt_xy.shape, gt_xy.dtype)   # [B,N,2]
            print("gt_xy min/max:", gt_xy[0].min(0).values, gt_xy[0].max(0).values)
            print("fragment size:", imgs_norm.shape[-2:], "(H,W)")
            gt_canvas = assemble_canvas(imgs_norm[0], gt_xy[0], gt_rot[0], pad_mask[0],img_sizes=img_sizes[0],canvas_size=(canvas_width,canvas_height))
            pr_canvas = assemble_canvas(imgs_norm[0], pred_xy[0], pred_rot[0], pad_mask[0],img_sizes=img_sizes[0],canvas_size=(canvas_width,canvas_height) )


            fig, axes = plt.subplots(1, 2, figsize=(12, 6))
            axes[0].imshow(np.asarray(gt_canvas))
            axes[0].set_title(f"{split_name}: Ground Truth Assembly")
            axes[0].axis("off")

            axes[1].imshow(np.asarray(pr_canvas))
            axes[1].set_title(f"{split_name}: Predicted Assembly")
            axes[1].axis("off")
            plt.tight_layout()
            plt.show()

            shown += 1
            if shown >= num_examples:
                break

# ----------------------------
# Render training + validation
# ----------------------------
model = PuzzlePoseModel().to(DEVICE)
model.load_state_dict("example_username-model.pth")
model.eval()
render_examples(train_loader, "TRAIN", NUM_TRAIN_EXAMPLES)
render_examples(val_loader, "VAL", NUM_VAL_EXAMPLES)

TypeError: Expected state_dict to be dict-like, got <class 'str'>.

## 6. Count Number of Parameters (Do not remove)
Before training, we compute how many trainable parameters the model has.

A parameter is a number the model learns during training (for example, weights in convolutional layers or attention layers).
Only parameters with requires_grad = True are updated by the optimizer. Some parameters may be frozen (not trained), and those are intentionally excluded here.

This count is important because:
 - It tells you how large your model is.
 - Larger models are harder to train and easier to overfit.
 - When you change the architecture (e.g., d_model, number of layers), this number should change.

When you change the input resolution (e.g., 128 → 24), the number of parameters does not change, which is an important concept to understand.

⚠️ Important:
If you freeze or unfreeze parameters after this cell runs, the reported number will be incorrect.
This cell must run after the model is fully constructed and before training begins.

In [10]:
def count_trainable_parameters(model: torch.nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

num_params = count_trainable_parameters(model)
print(f"Trainable parameters: {num_params:,}")

# Hard safety check for oversized models
MAX_PARAMS = 15_000_000

if num_params > MAX_PARAMS:
    print("\n" + "!" * 80)
    print("WARNING: MODEL IS TOO LARGE")
    print(f"This model has {num_params:,} trainable parameters.")
    print(f"The recommended maximum for this assignment is {MAX_PARAMS:,}.")
    print()
    print("Large models:")
    print("- Train much more slowly")
    print("- Are more likely to overfit")
    print("- May exceed memory or runtime limits")
    print()
    print("Consider reducing:")
    print("- d_model")
    print("- number of Transformer layers")
    print("- number of attention heads")
    print("!" * 80 + "\n")

Trainable parameters: 259,683


## 7. Time Script (Do not remove)

In [11]:
script_end = time.perf_counter()
total_time = script_end - script_start

print(f"Total execution time: {total_time:.6f} seconds")

Total execution time: 2968.166979 seconds
